In [0]:
dbutils.widgets.text("source_table", "workspace.cdf_benchmark_source.bookings")
dbutils.widgets.text("row_count", "10000000")

source_table = dbutils.widgets.get("source_table")
row_count = int(dbutils.widgets.get("row_count"))

In [0]:
%sql
CREATE TABLE IF NOT EXISTS IDENTIFIER(:source_table) (
  booking_id BIGINT,
  customer_id BIGINT,
  property_id BIGINT,
  booking_date DATE,
  checkin_date DATE,
  checkout_date DATE,
  amount DECIMAL(12,2),
  status STRING,
  updated_at TIMESTAMP
)
CLUSTER BY (booking_date)
TBLPROPERTIES (
  delta.enableChangeDataFeed = true
)

In [0]:
base_df = (
    spark.range(1, row_count + 1)
    .selectExpr(
        "id AS booking_id",
        "CAST((id % 1000000) + 1 AS BIGINT) AS customer_id",
        "CAST((id % 250000) + 1 AS BIGINT) AS property_id",
        "date_sub(date'2026-04-01', CAST(id % 365 AS INT)) AS booking_date",
        "date_add(date_sub(date'2026-04-01', CAST(id % 365 AS INT)), CAST(id % 30 AS INT)) AS checkin_date",
        "date_add(date_add(date_sub(date'2026-04-01', CAST(id % 365 AS INT)), CAST(id % 30 AS INT)), CAST((id % 10) + 1 AS INT)) AS checkout_date",
        "CAST(((id % 50000) / 10.0) + 50 AS DECIMAL(12,2)) AS amount",
        "CASE WHEN id % 5 = 0 THEN 'confirmed' WHEN id % 5 = 1 THEN 'pending' WHEN id % 5 = 2 THEN 'cancelled' WHEN id % 5 = 3 THEN 'checked_in' ELSE 'checked_out' END AS status",
        "TIMESTAMP'2026-04-01 00:00:00' AS updated_at"
    )
)

base_df.write.format("delta").mode("append").saveAsTable(source_table)

In [0]:
updates_df = (
    spark.table(source_table)
    .sample(False, 3000000)
    .selectExpr(
        "booking_id",
        "booking_date",
        "CAST(amount + 75.00 AS DECIMAL(12,2)) AS amount",
        "'confirmed' AS status",
        "current_timestamp() AS updated_at"
    )
)

updates_df.createOrReplaceTempView("updates")

spark.sql(f'''
MERGE INTO {source_table} t
USING updates s
ON t.booking_id = s.booking_id AND t.booking_date = s.booking_date
WHEN MATCHED THEN UPDATE SET
  t.amount = s.amount,
  t.status = s.status,
  t.updated_at = s.updated_at
''')

In [0]:
deletes_df = (
    spark.table(source_table)
    .sample(False, 2000000)
    .select("booking_id", "booking_date")
)

deletes_df.createOrReplaceTempView("deletes")

spark.sql(f'''
MERGE INTO {source_table} t
USING deletes s
ON t.booking_id = s.booking_id AND t.booking_date = s.booking_date
WHEN MATCHED THEN DELETE
''')

In [0]:
%sql
OPTIMIZE IDENTIFIER(:source_table) FULL;